# KKBox 고객 이탈 예측 — 모델 학습 및 비교

| 항목 | 내용 |
|------|------|
| 데이터 | train_split.pkl / val_split.pkl / test_split.pkl |
| 타깃 | `is_churn` (0=유지, 1=이탈) |
| 이탈율 | 약 9.46% (클래스 불균형) |
| 사용 모델 | Decision Tree / XGBoost / LightGBM |
| 튜닝 방법 | GridSearchCV (AUC 기준) |

**산출물 (results/)**
- `model_comparison.csv` — 모델 성능 비교표
- `dt_confusion_matrix.png` / `xgb_confusion_matrix.png` / `lgbm_confusion_matrix.png`
- `dt_feature_importance.png` / `xgb_feature_importance.png` / `lgbm_feature_importance.png`
- `dt_classification_report.csv` / `xgb_classification_report.csv` / `lgbm_classification_report.csv`
- 최종 모델: `shap_summary.png` / `shap_beeswarm.png` / `risk_decile_distribution.png`

## 0. 라이브러리 & 경로 설정

In [1]:
import os
import sys
import time
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
warnings.filterwarnings('ignore')

# src 경로 추가
sys.path.append('./src')

from data_loader import load_data, get_scale_pos_weight
from metrics import (
    evaluate_model,
    save_classification_report,
    save_confusion_matrix,
    save_roc_curve,
    save_roc_curve_combined,
    save_model_comparison,
)
from models.dt import tune_dt, train_dt, save_feature_importance as dt_importance
from models.xgboost_model import tune_xgb, train_xgb, save_feature_importance as xgb_importance
from models.lgbm_model import tune_lgbm, train_lgbm, save_feature_importance as lgbm_importance

DATA_DIR    = './data'
RESULTS_DIR = './results'
MODEL_DIR   = './model'
SEED        = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print('설정 완료')

설정 완료


## 1. 데이터 로드

In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data(DATA_DIR)

# 클래스 불균형 대응 가중치
scale_pos_weight = get_scale_pos_weight(y_train)
print(f'\nscale_pos_weight : {scale_pos_weight}')
print(f'feature 수       : {X_train.shape[1]}개')

데이터 로드 중...
  train : (516580, 42)
  val   : (172193, 42)
  test  : (172194, 42)

[클래스 분포]
  train: 유지=467,711  이탈=48,869  이탈율=9.46%
  val: 유지=155,904  이탈=16,289  이탈율=9.46%
  test: 유지=155,904  이탈=16,290  이탈율=9.46%

사용 피처 수: 32개
피처 목록: ['city', 'registered_via', 'reg_year', 'reg_month', 'reg_day', 'reg_weekday', 'reg_time_num', 'gender_female', 'gender_male', 'gender_unknown', 'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew', 'is_cancel', 'trans_day', 'trans_weekday', 'expire_day', 'expire_weekday', 'pm_id_41', 'pm_id_38', 'pp_days_30', 'trans_count', 'total_secs', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq', 'log_count']

scale_pos_weight : 9.5707
feature 수       : 32개


## 2. Decision Tree

### 2-1. 하이퍼파라미터 튜닝

| 파라미터 | 탐색 범위 | 선택 근거 |
|---|---|---|
| max_depth | 5, 7, 10, 15 | 깊을수록 과적합 위험 — val AUC로 최적 깊이 탐색 |
| min_samples_leaf | 10, 50, 100 | 리프 최소 샘플 수 제한으로 노이즈 과적합 방지 |
| min_samples_split | 20, 50, 100 | 분기 최소 샘플 수 제한으로 희귀 패턴 과적합 방지 |
| class_weight | balanced (고정) | 이탈율 9.46% 불균형 → 소수 클래스 자동 가중치 |

In [3]:
########## Decision Tree 튜닝 ##########
dt_best_params = tune_dt(X_train, y_train, cv=5)
print(f'\n선택된 파라미터: {dt_best_params}')

########## Decision Tree GridSearchCV 시작 ##########
Fitting 5 folds for each of 80 candidates, totalling 400 fits



최적 파라미터 : {'max_depth': 15, 'min_samples_leaf': 100, 'min_samples_split': 10}
최적 AUC(CV) : 0.9686
소요 시간    : 402.6s

선택된 파라미터: {'max_depth': 15, 'min_samples_leaf': 100, 'min_samples_split': 10}


### 2-2. 학습 및 평가

In [4]:
########## Decision Tree 학습 ##########
dt_model = train_dt(X_train, y_train, params=dt_best_params)

# 예측
dt_pred  = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

# 평가
dt_result = evaluate_model('Decision Tree', y_test, dt_pred, dt_proba)
print('\n[Decision Tree 성능]')
for k, v in dt_result.items():
    print(f'  {k:10s}: {v}')

# 산출물 저장
save_confusion_matrix('Decision Tree', y_test, dt_pred, RESULTS_DIR)
save_classification_report('Decision Tree', y_test, dt_pred, RESULTS_DIR)
dt_importance(dt_model, X_train.columns, RESULTS_DIR)
save_roc_curve('Decision Tree', y_test, dt_proba, RESULTS_DIR)


# 모델 저장
with open(os.path.join(MODEL_DIR, 'dt_model.pkl'), 'wb') as f:
    pickle.dump(dt_model, f)
print('  [저장] dt_model.pkl')


########## Decision Tree 학습 ##########
학습 완료 (10.4s)

[Decision Tree 성능]
  Model     : Decision Tree
  AUC       : 0.969114
  F1        : 0.685885
  Log Loss  : 0.215415
  Accuracy  : 0.921281
  Precision : 0.550906
  Recall    : 0.908471
  [저장] decision_tree_confusion_matrix.png
  [저장] decision_tree_classification_report.csv
  [저장] dt_feature_importance.csv / dt_feature_importance.png
  [저장] decision_tree_roc_curve.png
  [저장] dt_model.pkl


## 3. XGBoost

### 3-1. 하이퍼파라미터 튜닝

| 파라미터 | 탐색 범위 | 선택 근거 |
|---|---|---|
| max_depth | 4, 5, 6 | 깊을수록 복잡한 패턴 학습 but 과적합 위험 |
| learning_rate | 0.05, 0.1 | 낮을수록 안정적 수렴, 높을수록 빠른 학습 |
| subsample | 0.7, 0.8 | 행 샘플링으로 과적합 방지 및 다양성 확보 |
| colsample_bytree | 0.7, 0.8 | 피처 샘플링으로 특정 피처 의존도 완화 |
| scale_pos_weight | 9.57 (고정) | 유지/이탈 비율 → 소수 클래스 가중치 |

In [5]:
########## XGBoost 튜닝 ##########
xgb_best_params = tune_xgb(X_train, y_train, scale_pos_weight=scale_pos_weight, cv=5)
print(f'\n선택된 파라미터: {xgb_best_params}')

########## XGBoost GridSearchCV 시작 ##########
Fitting 5 folds for each of 24 candidates, totalling 120 fits

최적 파라미터 : {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'subsample': 0.8}
최적 AUC(CV) : 0.9899
소요 시간    : 504.4s

선택된 파라미터: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'subsample': 0.8}


### 3-2. 학습 및 평가

In [6]:
########## XGBoost 학습 ##########
xgb_model = train_xgb(X_train, y_train, X_val, y_val,
                       params=xgb_best_params, scale_pos_weight=scale_pos_weight)

# 예측
xgb_pred  = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

# 평가
xgb_result = evaluate_model('XGBoost', y_test, xgb_pred, xgb_proba)
print('\n[XGBoost 성능]')
for k, v in xgb_result.items():
    print(f'  {k:10s}: {v}')

# 산출물 저장
save_confusion_matrix('XGBoost', y_test, xgb_pred, RESULTS_DIR)
save_classification_report('XGBoost', y_test, xgb_pred, RESULTS_DIR)
xgb_importance(xgb_model, X_train.columns, RESULTS_DIR)
save_roc_curve('XGBoost', y_test, xgb_proba, RESULTS_DIR)


# 모델 저장
with open(os.path.join(MODEL_DIR, 'xgb_model.pkl'), 'wb') as f:
    pickle.dump(xgb_model, f)
print('  [저장] xgb_model.pkl')


########## XGBoost 학습 ##########
[0]	validation_0-logloss:0.63032
[100]	validation_0-logloss:0.15771
[200]	validation_0-logloss:0.13269
[300]	validation_0-logloss:0.12197
[400]	validation_0-logloss:0.11421
[500]	validation_0-logloss:0.10851
[600]	validation_0-logloss:0.10394
[700]	validation_0-logloss:0.10101
[800]	validation_0-logloss:0.09824
[900]	validation_0-logloss:0.09511
[999]	validation_0-logloss:0.09250
최적 트리 수 : 999
학습 완료   : 41.1s



[XGBoost 성능]
  Model     : XGBoost
  AUC       : 0.991799
  F1        : 0.856823
  Log Loss  : 0.091506
  Accuracy  : 0.970313
  Precision : 0.787885
  Recall    : 0.938981
  [저장] xgboost_confusion_matrix.png
  [저장] xgboost_classification_report.csv
  [저장] xgb_feature_importance.csv / xgb_feature_importance.png
  [저장] xgboost_roc_curve.png
  [저장] xgb_model.pkl


## 4. LightGBM

### 4-1. 하이퍼파라미터 튜닝

| 파라미터 | 탐색 범위 | 선택 근거 |
|---|---|---|
| num_leaves | 31, 63, 127 | max_depth 대신 리프 수로 복잡도 제어 (LightGBM 방식) |
| learning_rate | 0.05, 0.1 | 낮을수록 안정적 — early stopping으로 과학습 방지 |
| min_child_samples | 20, 50, 100 | 리프 최소 샘플 수 → 희귀 패턴 과적합 방지 |
| subsample | 0.7, 0.8 | 행 샘플링으로 과적합 방지 |
| scale_pos_weight | 9.57 (고정) | 클래스 불균형 대응 |
| early_stopping | 50라운드 (고정) | val logloss 기준 최적 트리 수 자동 탐색 |

In [7]:
########## LightGBM 튜닝 ##########
lgbm_best_params = tune_lgbm(X_train, y_train, scale_pos_weight=scale_pos_weight, cv=5)
print(f'\n선택된 파라미터: {lgbm_best_params}')

########## LightGBM GridSearchCV 시작 ##########
Fitting 5 folds for each of 36 candidates, totalling 180 fits

최적 파라미터 : {'learning_rate': 0.1, 'min_child_samples': 20, 'num_leaves': 127, 'subsample': 0.7}
최적 AUC(CV) : 0.9919
소요 시간    : 838.4s

선택된 파라미터: {'learning_rate': 0.1, 'min_child_samples': 20, 'num_leaves': 127, 'subsample': 0.7}


### 4-2. 학습 및 평가

In [8]:
########## LightGBM 학습 (Early Stopping) ##########
lgbm_model = train_lgbm(X_train, y_train, X_val, y_val,
                         params=lgbm_best_params, scale_pos_weight=scale_pos_weight)

# 예측
lgbm_pred  = lgbm_model.predict(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]

# 평가
lgbm_result = evaluate_model('LightGBM', y_test, lgbm_pred, lgbm_proba)
print('\n[LightGBM 성능]')
for k, v in lgbm_result.items():
    print(f'  {k:10s}: {v}')

# 산출물 저장
save_confusion_matrix('LightGBM', y_test, lgbm_pred, RESULTS_DIR)
save_classification_report('LightGBM', y_test, lgbm_pred, RESULTS_DIR)
lgbm_importance(lgbm_model, X_train.columns, RESULTS_DIR)
save_roc_curve('LightGBM', y_test, lgbm_proba, RESULTS_DIR)


# 모델 저장
lgbm_model.booster_.save_model(os.path.join(MODEL_DIR, 'lgbm_model.txt'))
print('  [저장] lgbm_model.txt')


########## LightGBM 학습 (Early Stopping) ##########
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.113547
[200]	valid_0's binary_logloss: 0.10064
[300]	valid_0's binary_logloss: 0.0929355
[400]	valid_0's binary_logloss: 0.0889886
[500]	valid_0's binary_logloss: 0.0847807
[600]	valid_0's binary_logloss: 0.0814481
[700]	valid_0's binary_logloss: 0.0784389
[800]	valid_0's binary_logloss: 0.0769863
[900]	valid_0's binary_logloss: 0.0762654
[1000]	valid_0's binary_logloss: 0.0755379
[1100]	valid_0's binary_logloss: 0.0752852
[1200]	valid_0's binary_logloss: 0.0752058
Early stopping, best iteration is:
[1164]	valid_0's binary_logloss: 0.0751504

최적 트리 수 : 1164
학습 완료   : 24.5s

[LightGBM 성능]
  Model     : LightGBM
  AUC       : 0.992738
  F1        : 0.875214
  Log Loss  : 0.073275
  Accuracy  : 0.975069
  Precision : 0.831171
  Recall    : 0.924187
  [저장] lightgbm_confusion_matrix.png
  [저장] lightgbm_classification_report.csv
  [저장] lgbm_featur

## 5. 모델 성능 비교

In [9]:
########## 모델 비교표 생성 ##########
records = [dt_result, xgb_result, lgbm_result]
comparison_df = save_model_comparison(records, RESULTS_DIR)

save_roc_curve_combined([
    {'name': 'Decision Tree', 'y_true': y_test, 'y_proba': dt_proba},
    {'name': 'XGBoost',       'y_true': y_test, 'y_proba': xgb_proba},
    {'name': 'LightGBM',      'y_true': y_test, 'y_proba': lgbm_proba},
], RESULTS_DIR)


           📊 모델 성능 비교
        Model      AUC       F1  Log Loss  Accuracy  Precision   Recall
     LightGBM 0.992738 0.875214  0.073275  0.975069   0.831171 0.924187
      XGBoost 0.991799 0.856823  0.091506  0.970313   0.787885 0.938981
Decision Tree 0.969114 0.685885  0.215415  0.921281   0.550906 0.908471
  [저장] model_comparison.csv
  [저장] roc_curve_combined.png


## 6. 최종 모델 선정

AUC 기준으로 가장 성능이 좋은 모델을 최종 선정

In [10]:
########## 최종 모델 선정 ##########
best_row   = comparison_df.iloc[0]
best_name  = best_row['Model']

model_map = {
    'Decision Tree': (dt_model,   dt_pred,   dt_proba),
    'XGBoost'      : (xgb_model,  xgb_pred,  xgb_proba),
    'LightGBM'     : (lgbm_model, lgbm_pred, lgbm_proba),
}

best_model, best_pred, best_proba = model_map[best_name]

print(f'최종 선정 모델 : {best_name}')
print(f'  AUC      : {best_row["AUC"]}')
print(f'  F1       : {best_row["F1"]}')
print(f'  Log Loss : {best_row["Log Loss"]}')

최종 선정 모델 : LightGBM
  AUC      : 0.992738
  F1       : 0.875214
  Log Loss : 0.073275


## 7. SHAP 분석 (최종 모델)

In [11]:
########## SHAP 분석 ##########
print('SHAP 계산 중 (1~2분 소요)...')
t0 = time.time()

N_SHAP  = min(5000, len(X_test))
X_shap  = X_test.sample(N_SHAP, random_state=SEED).reset_index(drop=True)

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap)

# LightGBM은 shap_values가 2D, 나머지도 동일하게 처리
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# SHAP 요약 CSV
shap_df = pd.DataFrame({
    'Feature'      : X_train.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

shap_df.to_csv(os.path.join(RESULTS_DIR, 'shap_summary.csv'), index=False, encoding='utf-8-sig')
print(f'  [저장] shap_summary.csv ({time.time()-t0:.1f}s)')
print(shap_df.head(10).to_string(index=False))

SHAP 계산 중 (1~2분 소요)...
  [저장] shap_summary.csv (51.1s)
          Feature  mean_abs_shap
       expire_day       0.810931
    is_auto_renew       0.751060
   expire_weekday       0.666224
        trans_day       0.637801
        is_cancel       0.396002
     reg_time_num       0.374745
payment_method_id       0.327530
      trans_count       0.303771
    trans_weekday       0.283424
             city       0.211226


In [12]:
# SHAP Bar Plot
plt.figure()
shap.summary_plot(shap_values, X_shap, plot_type='bar',
                  feature_names=list(X_train.columns), max_display=15, show=False)
plt.title(f'SHAP Feature Importance — {best_name}')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print('  [저장] shap_summary.png')

  [저장] shap_summary.png


In [13]:
# SHAP Beeswarm Plot
plt.figure()
shap.summary_plot(shap_values, X_shap,
                  feature_names=list(X_train.columns), max_display=15, show=False)
plt.title(f'SHAP Beeswarm — {best_name}')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'shap_beeswarm.png'), dpi=150, bbox_inches='tight')
plt.show()
print('  [저장] shap_beeswarm.png')

  [저장] shap_beeswarm.png


## 8. Risk Decile 분석 (최종 모델)

In [14]:
########## Risk Decile 분포 ##########
result = pd.DataFrame({
    'msno'             : X_test.index,
    'actual_churn'     : y_test.values,
    'predicted_churn'  : best_pred,
    'churn_probability': best_proba,
})

# 10분위 위험 등급 (1=최고위험, 10=최저위험)
result['risk_decile'] = pd.qcut(
    result['churn_probability'].rank(method='first', ascending=False),
    q=10, labels=list(range(1, 11))
).astype(int)

# 3단계 위험군
result['risk_group'] = result['risk_decile'].map(
    lambda d: 'High Risk' if d <= 3 else ('Mid Risk' if d <= 7 else 'Low Risk')
)

result = result.sort_values('churn_probability', ascending=False).reset_index(drop=True)
result.to_csv(os.path.join(RESULTS_DIR, 'churn_prediction_test.csv'), index=False, encoding='utf-8-sig')
print(f'  [저장] churn_prediction_test.csv ({len(result):,}명)')

print('\n--- 위험군별 실제 이탈율 ---')
print(result.groupby('risk_group')['actual_churn']
      .agg(['count', 'sum', 'mean'])
      .rename(columns={'count':'회원수','sum':'실제이탈수','mean':'실제이탈율'}))

  [저장] churn_prediction_test.csv (172,194명)

--- 위험군별 실제 이탈율 ---
              회원수  실제이탈수     실제이탈율
risk_group                        
High Risk   51658  16257  0.314704
Low Risk    51658      0  0.000000
Mid Risk    68878     33  0.000479


In [15]:
# Risk Decile 시각화
decile_summary = result.groupby('risk_decile').agg(
    count=('actual_churn', 'count'),
    churn_rate=('actual_churn', 'mean')
).reset_index()

colors = ['#d73027' if d <= 3 else '#fee090' if d <= 7 else '#91bfdb'
          for d in decile_summary['risk_decile']]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(decile_summary['risk_decile'], decile_summary['count'], color=colors, alpha=0.8)
ax2.plot(decile_summary['risk_decile'], decile_summary['churn_rate'], 'ko-', linewidth=2, markersize=6)

ax1.set_xlabel('Risk Decile  (1=최고위험 → 10=최저위험)')
ax1.set_ylabel('회원 수')
ax2.set_ylabel('실제 이탈율')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
ax1.set_xticks(range(1, 11))
plt.title(f'Risk Decile 분포 vs 실제 이탈율 — {best_name}')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'risk_decile_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('  [저장] risk_decile_distribution.png')

  [저장] risk_decile_distribution.png


## 9. 최종 산출물 확인

In [16]:
print('\n========== 최종 산출물 목록 ==========')
for f in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, f)
    print(f'  {f:50s}  {os.path.getsize(fpath)/1024:.1f} KB')
print('\n✅ 전체 완료')


========== 최종 산출물 목록 ==========
  churn_prediction_test.csv                           7524.7 KB
  decision_tree_classification_report.csv             0.4 KB
  decision_tree_confusion_matrix.png                  25.1 KB
  decision_tree_roc_curve.png                         54.1 KB
  dt_feature_importance.csv                           1.0 KB
  dt_feature_importance.png                           60.0 KB
  lgbm_feature_importance.csv                         0.6 KB
  lgbm_feature_importance.png                         61.0 KB
  lightgbm_classification_report.csv                  0.4 KB
  lightgbm_confusion_matrix.png                       24.8 KB
  lightgbm_roc_curve.png                              52.0 KB
  model_comparison.csv                                0.2 KB
  risk_decile_distribution.png                        52.2 KB
  roc_curve_combined.png                              74.2 KB
  shap_beeswarm.png                                   166.5 KB
  shap_summary.csv                     

In [17]:
########## 과적합 확인 — Train vs Val vs Test ##########
from sklearn.metrics import roc_auc_score

models = {
    'Decision Tree': (dt_model,   dt_pred,   dt_proba),
    'XGBoost'      : (xgb_model,  xgb_pred,  xgb_proba),
    'LightGBM'     : (lgbm_model, lgbm_pred, lgbm_proba),
}

print(f"{'모델':<15} {'Train AUC':>10} {'Val AUC':>10} {'Test AUC':>10} {'Train-Val':>10} {'판정':>8}")
print("-" * 65)

for name, (model, pred, proba) in models.items():
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
    test_auc  = roc_auc_score(y_test,  proba)
    diff      = train_auc - val_auc

    if diff < 0.01:
        judge = '✅ 정상'
    elif diff < 0.03:
        judge = '⚠️ 경미'
    else:
        judge = '❌ 과적합'

    print(f"{name:<15} {train_auc:>10.4f} {val_auc:>10.4f} {test_auc:>10.4f} {diff:>10.4f} {judge:>8}")

모델               Train AUC    Val AUC   Test AUC  Train-Val       판정
-----------------------------------------------------------------
Decision Tree       0.9782     0.9683     0.9691     0.0099     ✅ 정상
XGBoost             0.9978     0.9913     0.9918     0.0066     ✅ 정상
LightGBM            1.0000     0.9922     0.9927     0.0078     ✅ 정상
